# Fine-tune SmolVLA on a LeRobot dataset (Colab)

Fine-tune Hugging Face's SmolVLA (a 450M Vision-Language-Action model) on a LeRobot manipulation dataset, entirely in Google Colab (free T4 GPU).

> ⚠️ **LeRobot requires Python >= 3.12.** Check the first cell's output; if it is not 3.12, in Colab go to **Runtime → Change runtime type** and pick a newer runtime, or use the [official HF notebook](https://colab.research.google.com/github/huggingface/notebooks/blob/main/lerobot/training-smolvla.ipynb) which is kept up to date.

The pretrained base model is `lerobot/smolvla_base` (not `HuggingFaceTB/SmolVLA-Base`, which is gated).

In [ ]:
# @title 1. Install LeRobot + SmolVLA dependencies
import sys
print("Python version:", sys.version)  # must be >= 3.12

!git clone https://github.com/huggingface/lerobot.git
%cd lerobot
!pip install -q -e ".[smolvla]"
!pip install -q -U torchao  # fix torchao/diffusers version conflict on Colab

In [ ]:
# @title 2. Log in to the Hugging Face Hub
# Get a WRITE token at https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# @title 3. Fine-tune SmolVLA
# Replace YOUR_USERNAME. The dataset below is the SO-100 pick-place dataset
# from the SmolVLA paper; its cameras (top/wrist) are renamed to SmolVLA's
# camera1/camera2, and the third camera slot is left empty (empty_cameras=1).
!lerobot-train \
    --policy.path=lerobot/smolvla_base \
    --dataset.repo_id=lerobot/svla_so100_pickplace \
    --policy.repo_id=YOUR_USERNAME/my-smolvla \
    --policy.push_to_hub=false \
    --policy.empty_cameras=1 \
    --rename_map='{"observation.images.top": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}' \
    --batch_size=16 \
    --steps=2000 \
    --output_dir=outputs/train/my_smolvla \
    --job_name=my_smolvla_training \
    --policy.device=cuda \
    --wandb.enable=false

## What's happening & tuning tips

- `--policy.path=lerobot/smolvla_base` loads the pretrained 450M base model; training adapts it to your dataset.
- `--steps=2000` is a **smoke test**. The paper uses ~20,000 steps (~4 h on an A100; longer on the free T4).
- `--batch_size=16` is conservative for the T4's 16 GB; increase if you have room.
- The checkpoint is saved to `outputs/train/my_smolvla/`. To push it to the Hub so you can reuse it, see the next cell.

For the full list of fine-tuning options, run `!lerobot-train --help`.

In [ ]:
# @title 4. (Optional) Push your fine-tuned model to the Hub
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path="outputs/train/my_smolvla",
    repo_id="YOUR_USERNAME/my-smolvla",
    repo_type="model",
)

## Next steps

- **Evaluate** the model: `lerobot-eval --help`, or run inference on real/sim hardware (see the [official SmolVLA docs](https://huggingface.co/docs/lerobot/policies/smolvla)).
- **Record your own data** to make it your own task: `lerobot-record`.
- **Deploy via ROS 2**: see `docs/SETUP.md` in this repo.